# Stage 3 — CPU CatBoost (ordered boosting) and four competing combiners

**Question:** does a different boosting algorithm add signal, and which combiner generalises best?

- CatBoost on CPU uses *ordered boosting* on small data, a different algorithm from the GPU's plain boosting.
- LightGBM extra-trees variant, a regularised NN, 10 seeds for cheap models.
- Four combiners (probability blend, logit blend, logit + country offsets, country stacker) ranked by held-out logloss.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| prob_blend | 0.27746 | 0.109161 | 1101 |
| logit_blend | 0.27758 | 0.109260 | 1102 |

**Finding:** every score is an exact multiple of 1/10,086, so the leaderboard counts wrong rows on the full test set. This submission changed only 22 rows vs. stage 2; differences of a few rows are noise. CPU CatBoost was the best and fastest model, which shaped stage 4.

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [13]:
import os, glob, time, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(name)
    return os.path.abspath(hits[0]) if hits else None

def gpu_available():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
USE_GPU = gpu_available()

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'                  # quick smoke test
N_FOLDS = 5
SEEDS = [0] if FAST else [0, 1, 2, 3, 4]              # expensive models (CatBoost)
SEEDS_CHEAP = [0] if FAST else list(range(10))        # cheap models (LightGBM, LogReg, NN)
RUN = dict(catboost_gpu=True, catboost_cpu=True, catboost_v2=True,
           lightgbm=True, lightgbm_xt=True, logreg=True, nn=True)
PREVIOUS_BEST = 0.108765
# ---------------------------------------------------------------------------

print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| GPU:', USE_GPU, '| FAST:', FAST, '| CPUs:', os.cpu_count())

train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | GPU: True | FAST: False | CPUs: 4


## Data and features (same as v2)

In [14]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']   # `year` maps 1:1 to country, so it is dropped

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)

# Counts use train+test together but never the target, so they cannot leak.
both = pd.concat([X, X_test], ignore_index=True)
def count_of(cols):
    return both.groupby(cols)[cols[0]].transform('size').values

extra = pd.DataFrame({
    'age_cat': both['age_of_respondent'].astype(str),
    'hh_cat': both['household_size'].clip(1, 15).astype(str),
    'cnt_profile': count_of(CATS + NUMS),
    'cnt_ctry_edu_job': count_of(['country', 'education_level', 'job_type']),
    'cnt_ctry_loc_cell': count_of(['country', 'location_type', 'cellphone_access']),
    'cnt_ctry_age': count_of(['country', 'age_of_respondent']),
})
X2 = pd.concat([X, extra.iloc[:len(X)].reset_index(drop=True)], axis=1)
X2_test = pd.concat([X_test, extra.iloc[len(X):].reset_index(drop=True)], axis=1)
CATS2 = CATS + ['age_cat', 'hh_cat']
CNTS = ['cnt_profile', 'cnt_ctry_edu_job', 'cnt_ctry_loc_cell', 'cnt_ctry_age']

print(X2.shape, X2_test.shape, '| positive rate: %.4f' % y.mean())

(23524, 16) (10086, 16) | positive rate: 0.1408


## Cross-validation harness
Identical folds per seed for every model. A model can use `SEEDS` or the larger `SEEDS_CHEAP`; either way each row's OOF prediction comes only from models that never saw it.

In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y))
         for s in sorted(set(SEEDS) | set(SEEDS_CHEAP))}
OOF, TEST, SCORES = {}, {}, {}

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit_predict, seeds=SEEDS):
    t0 = time.time()
    oof = np.zeros((len(seeds), len(y)))
    tst = np.zeros(len(X_test))
    for k, s in enumerate(seeds):
        for tr_idx, va_idx in FOLDS[s]:
            p_va, p_te = fit_predict(tr_idx, va_idx, s)
            oof[k, va_idx] = p_va
            tst += p_te / (len(seeds) * N_FOLDS)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST[name] = oof.mean(0), tst
    SCORES[name] = dict(cv_error=np.mean(per_seed), cv_error_std=np.std(per_seed),
                        logloss=np.mean([log_loss(y, o) for o in oof]),
                        logloss_avg=log_loss(y, OOF[name]), minutes=(time.time() - t0) / 60)
    print(f"{name:14s} logloss {SCORES[name]['logloss']:.5f} (seed-avg {SCORES[name]['logloss_avg']:.5f}) | "
          f"CV error {np.mean(per_seed):.5f} ± {np.std(per_seed):.5f} | {SCORES[name]['minutes']:.1f} min")

## CatBoost — GPU, CPU (ordered boosting) and v2 features

In [16]:
from catboost import CatBoostClassifier

def make_catboost(params, seed, gpu):
    extra = dict(task_type='GPU', devices='0') if (gpu and USE_GPU) else dict(thread_count=-1)
    return CatBoostClassifier(**params, **extra, random_seed=seed, verbose=0, allow_writing_files=False)

CB_PARAMS = dict(iterations=300 if FAST else 1000, learning_rate=0.03, depth=6,
                 l2_leaf_reg=5, max_ctr_complexity=2, border_count=254)
CB2_PARAMS = dict(iterations=300 if FAST else 1200, learning_rate=0.03, depth=6,
                  l2_leaf_reg=8, max_ctr_complexity=2, border_count=254, random_strength=1.5)

def catboost_fp(gpu):
    def fp(tr_idx, va_idx, seed):
        m = make_catboost(CB_PARAMS, seed + (0 if gpu else 50), gpu)
        m.fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
        return m.predict_proba(X.iloc[va_idx])[:, 1], m.predict_proba(X_test)[:, 1]
    return fp

def catboost_v2_fp(tr_idx, va_idx, seed):
    m = make_catboost(CB2_PARAMS, seed + 100, gpu=True)
    m.fit(X2.iloc[tr_idx], y[tr_idx], cat_features=CATS2)
    return m.predict_proba(X2.iloc[va_idx])[:, 1], m.predict_proba(X2_test)[:, 1]

if RUN['catboost_gpu']:
    run_model('catboost_gpu', catboost_fp(gpu=True))
if RUN['catboost_cpu']:
    run_model('catboost_cpu', catboost_fp(gpu=False))
if RUN['catboost_v2']:
    run_model('catboost_v2', catboost_v2_fp)

catboost_gpu   logloss 0.27935 (seed-avg 0.27843) | CV error 0.11167 ± 0.00020 | 15.4 min
catboost_cpu   logloss 0.27922 (seed-avg 0.27797) | CV error 0.11154 ± 0.00033 | 5.6 min
catboost_v2    logloss 0.27913 (seed-avg 0.27815) | CV error 0.11240 ± 0.00052 | 20.0 min


## LightGBM — standard and extra-trees variant

In [17]:
import lightgbm as lgb

def as_category(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=sorted(set(X2[c]) | set(X2_test[c])))
    return df

LGB_COLS = CATS + NUMS + CNTS
Xc, Xc_test = as_category(X2[LGB_COLS]), as_category(X2_test[LGB_COLS])

LGB_PARAMS = dict(n_estimators=150 if FAST else 700, learning_rate=0.02, num_leaves=24, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, cat_smooth=20, cat_l2=10,
                  reg_lambda=3, verbose=-1, n_jobs=-1)
LGB_XT_PARAMS = dict(LGB_PARAMS, n_estimators=150 if FAST else 1200, extra_trees=True,
                     num_leaves=31, min_child_samples=25)

def lgb_fp(params):
    def fp(tr_idx, va_idx, seed):
        m = lgb.LGBMClassifier(**params, random_state=seed)
        m.fit(Xc.iloc[tr_idx], y[tr_idx])
        return m.predict_proba(Xc.iloc[va_idx])[:, 1], m.predict_proba(Xc_test)[:, 1]
    return fp

if RUN['lightgbm']:
    run_model('lightgbm', lgb_fp(LGB_PARAMS), SEEDS_CHEAP)
if RUN['lightgbm_xt']:
    run_model('lightgbm_xt', lgb_fp(LGB_XT_PARAMS), SEEDS_CHEAP)

lightgbm       logloss 0.28158 (seed-avg 0.27988) | CV error 0.11274 ± 0.00063 | 1.3 min
lightgbm_xt    logloss 0.28171 (seed-avg 0.28013) | CV error 0.11372 ± 0.00054 | 2.6 min


## Logistic regression on pairwise + three-way crosses
v2 picked `C=0.02`, the lowest value tried, so the grid now extends below it.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

TRIPLES = [('country', 'education_level', 'job_type'),
           ('country', 'location_type', 'cellphone_access'),
           ('country', 'age_b', 'gender_of_respondent'),
           ('country', 'education_level', 'age_b'),
           ('country', 'relationship_with_head', 'marital_status')]

def crosses(df):
    d = df[CATS].copy()
    d['age_b'] = pd.cut(df['age_of_respondent'], [0, 20, 25, 30, 35, 40, 50, 60, 70, 200]).astype(str)
    d['hh_b'] = df['household_size'].clip(1, 12).astype(str)
    base = list(d.columns)
    for i in range(len(base)):
        for j in range(i + 1, len(base)):
            d[base[i] + '*' + base[j]] = d[base[i]] + '|' + d[base[j]]
    for a, b, c in TRIPLES:
        d[f'{a}*{b}*{c}'] = d[a] + '|' + d[b] + '|' + d[c]
    return d

Xx, Xx_test = crosses(X), crosses(X_test)

def make_lr_fp(C):
    def lr_fp(tr_idx, va_idx, seed):
        enc = OneHotEncoder(handle_unknown='ignore', min_frequency=5)
        A = enc.fit_transform(Xx.iloc[tr_idx])
        m = LogisticRegression(C=C, max_iter=3000)
        m.fit(A, y[tr_idx])
        return m.predict_proba(enc.transform(Xx.iloc[va_idx]))[:, 1], m.predict_proba(enc.transform(Xx_test))[:, 1]
    return lr_fp

if RUN['logreg']:
    grid = {}
    for C in [0.006, 0.01, 0.014, 0.02, 0.03]:
        fp = make_lr_fp(C)
        p = np.zeros(len(y))
        for tr_idx, va_idx in FOLDS[SEEDS[0]]:
            p[va_idx] = fp(tr_idx, va_idx, SEEDS[0])[0]
        grid[C] = log_loss(y, p)
    LR_C = min(grid, key=grid.get)
    print('LR grid (logloss):', {k: round(v, 5) for k, v in grid.items()}, '-> C =', LR_C)
    run_model('logreg_cross', make_lr_fp(LR_C), SEEDS_CHEAP)

LR grid (logloss): {0.006: 0.28241, 0.01: 0.2807, 0.014: 0.28019, 0.02: 0.28011, 0.03: 0.28052} -> C = 0.02
logreg_cross   logloss 0.28022 (seed-avg 0.27957) | CV error 0.11204 ± 0.00054 | 1.4 min


## Neural net with entity embeddings (GPU)
v2's net overfit (logloss 0.2845). v3: fewer epochs, more weight decay, smaller hidden layer.

In [19]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_COLS = CATS2
NN_EPOCHS, NN_BS = (3 if FAST else 10), 256

categories = {c: pd.Categorical(pd.concat([X2[c], X2_test[c]])).categories for c in EMB_COLS}

def nn_encode(df):
    cat = np.column_stack([pd.Categorical(df[c], categories=categories[c]).codes for c in EMB_COLS]).astype(np.int64)
    num = np.column_stack([df['age_of_respondent'], df['household_size']] +
                          [np.log1p(df[c]) for c in CNTS]).astype(np.float32)
    return cat, num

N_cat, N_num = nn_encode(X2)
T_cat, T_num = nn_encode(X2_test)
mu, sd = np.vstack([N_num, T_num]).mean(0), np.vstack([N_num, T_num]).std(0) + 1e-6
N_num, T_num = (N_num - mu) / sd, (T_num - mu) / sd

class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(len(categories[c]), min(8, (len(categories[c]) + 1) // 2 + 1))
                                   for c in EMB_COLS])
        d = sum(e.embedding_dim for e in self.embs) + N_num.shape[1]
        self.mlp = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(d, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.35),
            nn.Linear(128, 32), nn.BatchNorm1d(32), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(32, 1))

    def forward(self, c, x):
        h = torch.cat([e(c[:, i]) for i, e in enumerate(self.embs)] + [x], dim=1)
        return self.mlp(h).squeeze(1)

def to_t(a):
    return torch.from_numpy(a).to(DEVICE)

@torch.no_grad()
def nn_predict(model, cat, num):
    model.eval()
    return torch.sigmoid(model(to_t(cat), to_t(num))).cpu().numpy()

def nn_fp(tr_idx, va_idx, seed):
    torch.manual_seed(seed)
    model = EmbNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    steps = len(tr_idx) // NN_BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=NN_EPOCHS * steps)
    c_tr, x_tr = to_t(N_cat[tr_idx]), to_t(N_num[tr_idx])
    y_tr = torch.tensor(y[tr_idx], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    g = torch.Generator(device='cpu').manual_seed(seed)
    for _ in range(NN_EPOCHS):
        model.train()
        perm = torch.randperm(len(tr_idx), generator=g).to(DEVICE)
        for b in range(steps):                       # drops the last partial batch (BatchNorm-safe)
            i = perm[b * NN_BS:(b + 1) * NN_BS]
            opt.zero_grad()
            loss_fn(model(c_tr[i], x_tr[i]), y_tr[i]).backward()
            opt.step()
            sched.step()
    return nn_predict(model, N_cat[va_idx], N_num[va_idx]), nn_predict(model, T_cat, T_num)

if RUN['nn']:
    print('NN device:', DEVICE)
    run_model('embed_nn', nn_fp, SEEDS_CHEAP)

NN device: cuda
embed_nn       logloss 0.28405 (seed-avg 0.28128) | CV error 0.11361 ± 0.00050 | 2.0 min


## Save every model's predictions
So future versions can add models without retraining these.

In [20]:
pd.DataFrame({'y': y, **OOF}).to_csv(f'{OUT_DIR}/oof_v3.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **TEST}).to_csv(f'{OUT_DIR}/test_probabilities_v3.csv', index=False)
print('saved', list(OOF))

saved ['catboost_gpu', 'catboost_cpu', 'catboost_v2', 'lightgbm', 'lightgbm_xt', 'logreg_cross', 'embed_nn']


## Combiners — four candidates, judged on held-out logloss
Each combiner is fitted on 4/5 of the OOF rows and scored on the other 1/5 (repeated over 3 different splits), so none of them is graded on rows it was fitted on.

In [21]:
from scipy.optimize import minimize

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
T = np.column_stack([TEST[n] for n in names])

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

countries = sorted(train['country'].unique())
def onehot_country(ctry):
    return np.column_stack([(ctry == c).astype(float) for c in countries])

Z, Zt = logit(P), logit(T)
Cm, Ct = onehot_country(train['country'].values), onehot_country(test['country'].values)
inter = lambda Zm, Cmat: np.column_stack([Zm[:, j] * Cmat[:, k] for j in range(Zm.shape[1]) for k in range(Cmat.shape[1])])

class ProbBlend:                       # v2's winner: weighted average of probabilities
    def fit(self, F, t):
        def loss(w):
            w = np.abs(w) / np.abs(w).sum()
            return log_loss(t, np.clip(F @ w, 1e-6, 1 - 1e-6))
        r = minimize(loss, np.ones(F.shape[1]) / F.shape[1], method='Nelder-Mead',
                     options=dict(maxiter=6000, xatol=1e-5, fatol=1e-7))
        self.w = np.abs(r.x) / np.abs(r.x).sum()
        return self
    def predict(self, F):
        return F @ self.w

class LR:
    def __init__(self, C): self.C = C
    def fit(self, F, t):
        self.m = LogisticRegression(C=self.C, max_iter=5000).fit(F, t); return self
    def predict(self, F):
        return self.m.predict_proba(F)[:, 1]

CANDIDATES = {   # name: (combiner, train features, test features)
    'prob_blend':    (lambda: ProbBlend(), P, T),
    'logit_blend':   (lambda: LR(1.0), Z, Zt),
    'logit_country': (lambda: LR(1.0), np.hstack([Z, Cm]), np.hstack([Zt, Ct])),
    'country_stack': (lambda: LR(0.5), np.hstack([Z, Cm, inter(Z, Cm)]), np.hstack([Zt, Ct, inter(Zt, Ct)])),
}

HELD = {k: [] for k in CANDIDATES}
HELD_OOF = {k: np.zeros(len(y)) for k in CANDIDATES}
META_SEEDS = [2024, 7, 99]
for ms in META_SEEDS:
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=ms).split(P, y):
        for k, (make, F, _) in CANDIDATES.items():
            HELD_OOF[k][va] += make().fit(F[tr], y[tr]).predict(F[va]) / len(META_SEEDS)
for k in CANDIDATES:
    print(f'{k:14s} held-out logloss {log_loss(y, HELD_OOF[k]):.5f} | error {error_rate(HELD_OOF[k]):.5f}')

FITTED = {k: make().fit(F, y) for k, (make, F, _) in CANDIDATES.items()}
FINAL = {k: FITTED[k].predict(CANDIDATES[k][2]) for k in CANDIDATES}
ranked = sorted(CANDIDATES, key=lambda k: log_loss(y, HELD_OOF[k]))
print('\nranking by held-out logloss:', ranked)
print('prob_blend weights:', dict(zip(names, FITTED['prob_blend'].w.round(3))))

prob_blend     held-out logloss 0.27746 | error 0.11176
logit_blend    held-out logloss 0.27758 | error 0.11172
logit_country  held-out logloss 0.27773 | error 0.11180
country_stack  held-out logloss 0.27827 | error 0.11227

ranking by held-out logloss: ['prob_blend', 'logit_blend', 'logit_country', 'country_stack']
prob_blend weights: {'catboost_gpu': np.float64(0.16), 'catboost_cpu': np.float64(0.349), 'catboost_v2': np.float64(0.034), 'lightgbm': np.float64(0.135), 'lightgbm_xt': np.float64(0.116), 'logreg_cross': np.float64(0.126), 'embed_nn': np.float64(0.08)}


## Cutoff check
Only moves away from 0.5 if the gain beats a noise floor of 0.001 (~24 training rows).

In [22]:
grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
THRESH = {}
for k in CANDIDATES:
    o = HELD_OOF[k]
    errs = np.array([error_rate(o, t) for t in grid])
    THRESH[k] = grid[errs.argmin()] if errs.min() < error_rate(o) - 0.001 else 0.5
    print(f'{k:14s} best cutoff {grid[errs.argmin()]} (err {errs.min():.5f}) -> using {THRESH[k]}')

prob_blend     best cutoff 0.58 (err 0.11155) -> using 0.5
logit_blend    best cutoff 0.49 (err 0.11163) -> using 0.5
logit_country  best cutoff 0.55 (err 0.11167) -> using 0.5
country_stack  best cutoff 0.56 (err 0.11184) -> using 0.5


## Leaderboard of models and combiners (sorted by logloss, our best LB predictor)

In [23]:
board = pd.DataFrame(SCORES).T[['logloss', 'logloss_avg', 'cv_error', 'cv_error_std', 'minutes']]
for k in CANDIDATES:
    board.loc['>> ' + k] = [np.nan, log_loss(y, HELD_OOF[k]), error_rate(HELD_OOF[k], THRESH[k]), np.nan, np.nan]
display(board.sort_values('logloss_avg').round(5))
print('v2 blend (LB 0.108765) had held-out logloss 0.27754 - beat that to expect a better LB.')

,logloss,logloss_avg,cv_error,cv_error_std,minutes
>> prob_blend,NaN,0.27746,0.11176,NaN,NaN
>> logit_blend,NaN,0.27758,0.11172,NaN,NaN
>> logit_country,NaN,0.27773,0.11180,NaN,NaN
catboost_cpu,0.27922,0.27797,0.11154,0.00033,5.64074
catboost_v2,0.27913,0.27815,0.11240,0.00052,20.04497
>> country_stack,NaN,0.27827,0.11227,NaN,NaN
catboost_gpu,0.27935,0.27843,0.11167,0.00020,15.39964
logreg_cross,0.28022,0.27957,0.11204,0.00054,1.36580
lightgbm,0.28158,0.27988,0.11274,0.00063,1.29760
lightgbm_xt,0.28171,0.28013,0.11372,0.00054,2.61935


v2 blend (LB 0.108765) had held-out logloss 0.27754 - beat that to expect a better LB.


## Write submissions
`submission_v3_best.csv` = the combiner with the lowest held-out logloss. `submission_v3_second.csv` = runner-up, as a leaderboard sanity check.

In [24]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_v3_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')
    return sub

best, second = ranked[0], ranked[1]
print('best =', best, '| second =', second)
sub_best = save('best', FINAL[best], THRESH[best])
save('second', FINAL[second], THRESH[second])
for k in CANDIDATES:
    save(k, FINAL[k], THRESH[k])

prev_path = find_file('submission_v2_recommended.csv') or find_file('submission_v2_blend.csv')
if prev_path:
    prev = pd.read_csv(prev_path).set_index('unique_id')['bank_account']
    changed = (sub_best.set_index('unique_id')['bank_account'] != prev.reindex(sub_best['unique_id']).values).sum()
    print(f'best: {changed} rows differ from the v2 submission ({prev_path})')

best = prob_blend | second = logit_blend
/kaggle/working/submission_v3_best.csv: 757 ones / 10086 rows
/kaggle/working/submission_v3_second.csv: 770 ones / 10086 rows
/kaggle/working/submission_v3_prob_blend.csv: 757 ones / 10086 rows
/kaggle/working/submission_v3_logit_blend.csv: 770 ones / 10086 rows
/kaggle/working/submission_v3_logit_country.csv: 767 ones / 10086 rows
/kaggle/working/submission_v3_country_stack.csv: 766 ones / 10086 rows
best: 22 rows differ from the v2 submission (/kaggle/working/submission_v2_recommended.csv)
